# Notebook 4 — Paper Figures & Report
**App2Spec: RAG-Based Floating-Point Precision Recommendation**

Generates all publication-ready figures and tables for the conference paper:
1. Model accuracy comparison chart (all 7 models, 2 conditions)
2. RAG lift heatmap by domain
3. Precision class accuracy chart
4. Latency comparison chart
5. McNemar effect size chart
6. Three-way ablation chart (Zero-shot / Base-RAG / RAG+Metadata)
7. Pipeline architecture diagram (App2Spec vs MimirRAG comparison)
8. Representative RAG output example
9. Environment table
10. Full results summary for professor review

All figures saved to `results/figures/` as PDF and PNG.

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

PROJECT_ROOT = Path(r'C:\Users\chka24ac\App2spec')
RESULTS_ROOT = PROJECT_ROOT / 'results'
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
META_DIR     = RESULTS_ROOT / 'rag_metadata'
FIG_DIR      = RESULTS_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Publication style
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize':10,
    'ytick.labelsize':10,
    'legend.fontsize':10,
    'figure.dpi':     150,
    'savefig.dpi':    300,
    'savefig.bbox':   'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

COLORS = {
    'zeroshot': '#6B7280',
    'rag':      '#2563EB',
    'meta':     '#059669',
    'baseline': '#DC2626',
    'accent':   '#7C3AED',
}

def save_fig(name):
    plt.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight', dpi=300)
    plt.close()
    print(f'Saved: figures/{name}.pdf + .png')

print('Setup complete. Figures will be saved to:', FIG_DIR)

## 1. Load All Analysis Data

In [ ]:
df_accuracy  = pd.read_csv(ANALYSIS_DIR / 'accuracy_summary.csv')
df_mcnemar   = pd.read_csv(ANALYSIS_DIR / 'mcnemar_results.csv')
df_domain    = pd.read_csv(ANALYSIS_DIR / 'domain_accuracy.csv')
df_latency   = pd.read_csv(ANALYSIS_DIR / 'latency_summary.csv')
df_precision = pd.read_csv(ANALYSIS_DIR / 'precision_class_accuracy.csv')

# Load RAG+Metadata results
df_meta = pd.read_csv(META_DIR / 'rag_metadata_results.csv')
meta_summary = json.loads((META_DIR / 'summary.json').read_text())

# Model order (largest to smallest active params)
MODEL_ORDER = [
    'DeepSeek V4 Pro',
    'MiMo V2.5 Pro',
    'DeepSeek V3.2',
    'MiniMax M3',
    'DeepSeek V4 Flash',
    'gpt-oss-120b',
    'gpt-oss-20b (SLM)',
]

df_accuracy['Model'] = df_accuracy['Model'].str.strip()
df_acc = df_accuracy.set_index('Model').reindex(MODEL_ORDER).reset_index()

print('Data loaded.')
print(f'Models: {len(df_acc)}')
print(f'Domains: {df_domain["Domain Family"].nunique()}')
print(f'RAG+Meta scenarios: {len(df_meta)}')

## 2. Figure 1 — Model Accuracy Comparison

Grouped bar chart showing zero-shot and RAG accuracy for all 7 models, ordered by active parameters.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x      = np.arange(len(df_acc))
width  = 0.35

zs_vals  = df_acc['ZS Accuracy'].str.rstrip('%').astype(float)
rag_vals = df_acc['RAG Accuracy'].str.rstrip('%').astype(float)

bars_zs  = ax.bar(x - width/2, zs_vals,  width, label='Zero-shot',
                   color=COLORS['zeroshot'], alpha=0.85, edgecolor='white', linewidth=0.5)
bars_rag = ax.bar(x + width/2, rag_vals, width, label='Base-RAG',
                   color=COLORS['rag'],      alpha=0.85, edgecolor='white', linewidth=0.5)

# Annotate bars
for bar in bars_zs:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8, color='#374151')
for bar in bars_rag:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8, color='#1e40af')

# Baseline reference line
baseline_rag = rag_vals[df_acc['Model'] == 'DeepSeek V3.2'].values[0]
ax.axhline(baseline_rag, color=COLORS['baseline'], linestyle='--', linewidth=1.2, alpha=0.7,
           label=f'Baseline RAG ({baseline_rag:.1f}%)')

labels = [
    'DeepSeek V4 Pro\n(49B active)', 'MiMo V2.5 Pro\n(42B active)',
    'DeepSeek V3.2\n(37B active, baseline)', 'MiniMax M3\n(~15B active)',
    'DeepSeek V4 Flash\n(13B active)', 'gpt-oss-120b\n(5.1B active)',
    'gpt-oss-20b\n(3.6B active, SLM)',
]

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Zero-shot vs. Base-RAG Accuracy Across All Models\n(ordered by active parameters, largest to smallest)')
ax.set_ylim(40, 95)
ax.legend(loc='lower right')
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
save_fig('fig1_model_accuracy_comparison')

## 3. Figure 2 — RAG Lift Heatmap by Domain

Shows how much RAG improves over zero-shot for each model and domain family.

In [ ]:
pivot = df_domain.pivot_table(
    index='Domain Family', columns='Model', values='RAG Lift pp', aggfunc='mean'
)

# Reorder columns by active params
col_map = {
    'DeepSeek V4 Pro': 'V4 Pro\n(49B)',
    'MiMo V2.5 Pro':   'MiMo\n(42B)',
    'DeepSeek V3.2':   'V3.2\n(37B)',
    'MiniMax M3':      'M3\n(15B)',
    'DeepSeek V4 Flash':'V4 Flash\n(13B)',
    'gpt-oss-120b':    'gpt-120b\n(5.1B)',
    'gpt-oss-20b (SLM)':'gpt-20b\n(3.6B)',
}
pivot = pivot[[c for c in MODEL_ORDER if c in pivot.columns]]
pivot.columns = [col_map.get(c, c) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='RdYlGn',
    center=0, vmin=-20, vmax=80,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'RAG Lift (pp)', 'shrink': 0.8},
    ax=ax
)
ax.set_title('RAG Lift over Zero-shot by Domain and Model (percentage points)')
ax.set_xlabel('Model (active parameters)')
ax.set_ylabel('Benchmark Domain')
plt.tight_layout()
save_fig('fig2_rag_lift_heatmap')

## 4. Figure 3 — Precision Class Accuracy

Shows which precision formats are hardest to predict, averaged across all models.

In [ ]:
prec_avg = df_precision.groupby('Precision').agg(
    N=('N', 'first'),
    ZS_mean=('ZS Accuracy', 'mean'),
    RAG_mean=('RAG Accuracy', 'mean'),
).reset_index()
prec_avg = prec_avg[prec_avg['N'] >= 4].sort_values('RAG_mean', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(prec_avg))
width = 0.35

ax.barh(y - width/2, prec_avg['ZS_mean'] * 100,  width,
        color=COLORS['zeroshot'], alpha=0.85, label='Zero-shot')
ax.barh(y + width/2, prec_avg['RAG_mean'] * 100, width,
        color=COLORS['rag'],      alpha=0.85, label='Base-RAG')

for i, row in prec_avg.reset_index().iterrows():
    ax.text(row['ZS_mean']  * 100 + 0.5, i - width/2, f"{row['ZS_mean']*100:.1f}%",
            va='center', fontsize=8.5)
    ax.text(row['RAG_mean'] * 100 + 0.5, i + width/2, f"{row['RAG_mean']*100:.1f}%",
            va='center', fontsize=8.5, color='#1e40af')

ax.set_yticks(y)
ax.set_yticklabels([f"{r['Precision']} (n={r['N']})" for _, r in prec_avg.iterrows()])
ax.set_xlabel('Accuracy (%) — averaged across all 7 models')
ax.set_title('Accuracy by Precision Class: Zero-shot vs Base-RAG')
ax.legend(loc='lower right')
ax.xaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
save_fig('fig3_precision_class_accuracy')

## 5. Figure 4 — Latency Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: ZS vs RAG mean latency
ax = axes[0]
x = np.arange(len(df_latency))
w = 0.35
ax.bar(x - w/2, df_latency['ZS Mean (s)'],  w, color=COLORS['zeroshot'], alpha=0.85, label='Zero-shot')
ax.bar(x + w/2, df_latency['RAG Mean (s)'], w, color=COLORS['rag'],      alpha=0.85, label='Base-RAG')
ax.set_xticks(x)
ax.set_xticklabels(df_latency['Model'], rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('Mean Latency (s)')
ax.set_title('Mean API Latency per Scenario')
ax.legend()
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Right: P95 latency
ax = axes[1]
ax.bar(x - w/2, df_latency['ZS P95 (s)'],  w, color=COLORS['zeroshot'], alpha=0.85, label='Zero-shot P95')
ax.bar(x + w/2, df_latency['RAG P95 (s)'], w, color=COLORS['rag'],      alpha=0.85, label='Base-RAG P95')
ax.set_xticks(x)
ax.set_xticklabels(df_latency['Model'], rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('P95 Latency (s)')
ax.set_title('P95 Latency per Scenario (95th percentile)')
ax.legend()
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.suptitle('Latency Analysis: Zero-shot vs Base-RAG', fontsize=13, y=1.02)
plt.tight_layout()
save_fig('fig4_latency_comparison')

## 6. Figure 5 — McNemar Effect Size (Cohen's g)

In [ ]:
df_mc = df_mcnemar.copy()
df_mc['Model'] = df_mc['Model'].str.strip()
df_mc = df_mc.set_index('Model').reindex(MODEL_ORDER).reset_index()

colors_bar = [COLORS['accent'] if g >= 0.3 else COLORS['rag'] if g >= 0.1 else COLORS['zeroshot']
              for g in df_mc["Cohen's g"]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(df_mc)), df_mc["Cohen's g"], color=colors_bar, alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, df_mc.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{row[\"Cohen's g\"]:.3f}\n{row['Significance']}",
            ha='center', va='bottom', fontsize=8)

ax.axhline(0.3, color=COLORS['accent'], linestyle='--', linewidth=1, alpha=0.6, label='Large effect (g=0.3)')
ax.axhline(0.1, color=COLORS['rag'],    linestyle='--', linewidth=1, alpha=0.6, label='Medium effect (g=0.1)')
ax.set_xticks(range(len(df_mc)))
ax.set_xticklabels(df_mc['Model'], rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel("Cohen's g Effect Size")
ax.set_title("McNemar's Test: Effect Size of RAG vs Zero-shot (all p < 0.001)")
ax.set_ylim(0, 0.45)
ax.legend(loc='upper right')
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
save_fig('fig5_mcnemar_effect_size')

## 7. Figure 6 — Three-Way Ablation (Zero-shot / Base-RAG / RAG+Metadata)

For DeepSeek V4 Flash — the model used for the metadata condition.

In [ ]:
scored = df_meta[df_meta['error'].isna() | (df_meta['error'] == '')]

zs_acc   = scored['zeroshot_match'].mean() * 100
rag_acc  = scored['rag_match'].mean() * 100
meta_acc = scored['rag_meta_match'].mean() * 100

# Per-domain ablation
def map_domain(d):
    for key in ['AIBench','SuperBench','LLM Evaluation','DAWNBench','SPECfp 2017','TensorRT','MLPerf','HPC']:
        if str(d).startswith(key): return key
    return 'Other'

scored = scored.copy()
scored['domain_family'] = scored['domain'].apply(map_domain)

abl = scored.groupby('domain_family').agg(
    zs   = ('zeroshot_match', 'mean'),
    rag  = ('rag_match', 'mean'),
    meta = ('rag_meta_match', 'mean'),
    n    = ('zeroshot_match', 'count'),
).reset_index()
abl = abl[abl['n'] >= 3].sort_values('rag', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Overall three-way bar
ax = axes[0]
conditions = ['Zero-shot', 'Base-RAG', 'RAG+Metadata']
values     = [zs_acc, rag_acc, meta_acc]
bar_colors = [COLORS['zeroshot'], COLORS['rag'], COLORS['meta']]
bars = ax.bar(conditions, values, color=bar_colors, alpha=0.85, width=0.5, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Three-Way Comparison\n(DeepSeek V4 Flash, n=457)')
ax.set_ylim(40, 95)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Right: Per-domain comparison
ax = axes[1]
y = np.arange(len(abl))
w = 0.25
ax.barh(y - w,   abl['zs']   * 100, w, color=COLORS['zeroshot'], alpha=0.85, label='Zero-shot')
ax.barh(y,       abl['rag']  * 100, w, color=COLORS['rag'],      alpha=0.85, label='Base-RAG')
ax.barh(y + w,   abl['meta'] * 100, w, color=COLORS['meta'],     alpha=0.85, label='RAG+Metadata')
ax.set_yticks(y)
ax.set_yticklabels([f"{r['domain_family']} (n={r['n']})" for _, r in abl.iterrows()], fontsize=9)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Per-Domain: Three-Way Comparison')
ax.legend(loc='lower right')
ax.xaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.suptitle('Ablation Study: Zero-shot vs Base-RAG vs RAG+Metadata', fontsize=13, y=1.02)
plt.tight_layout()
save_fig('fig6_threeway_ablation')

## 8. Figure 7 — Pipeline Architecture Diagram

Compares App2Spec pipeline to MimirRAG architecture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def draw_box(ax, x, y, w, h, text, color, fontsize=9, text_color='white'):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='white', linewidth=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, color=text_color, fontweight='bold', zorder=3, wrap=True,
            multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2, color='#374151'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# ---- App2Spec Pipeline (left) ----
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('App2Spec Pipeline', fontsize=13, fontweight='bold', pad=15)

# Pre-retrieval
ax.text(5, 11.5, 'PRE-RETRIEVAL', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 0.5, 10.0, 4.0, 1.0, 'RAG4SPEC Excel\n(Sheet2 knowledge base)', '#1D4ED8')
draw_box(ax, 5.5, 10.0, 4.0, 1.0, 'all-MiniLM-L6-v2\nEmbedding Model', '#1D4ED8')
draw_box(ax, 3.0, 8.5,  4.0, 1.0, 'LanceDB\nVector Store', '#1E40AF')
draw_arrow(ax, 2.5, 10.0, 3.5, 9.5)
draw_arrow(ax, 7.5, 10.0, 6.5, 9.5)

# Retrieval
ax.text(5, 8.0, 'RETRIEVAL', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 0.5, 6.5, 3.5, 1.0, 'User Query\n(Workload Description)', '#059669')
draw_box(ax, 4.5, 6.5, 5.0, 1.0, 'Semantic Search\n(top-k=5, source exclusion)', '#047857')
draw_arrow(ax, 4.0, 7.0, 4.5, 7.0)
draw_arrow(ax, 5.0, 8.5, 5.0, 7.5)

# Conditions
ax.text(5, 6.0, 'GENERATION CONDITIONS', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 0.5, 4.5, 2.5, 1.0, 'Zero-shot\n(no context)', '#6B7280')
draw_box(ax, 3.5, 4.5, 2.5, 1.0, 'Base-RAG\n(retrieved context)', '#2563EB')
draw_box(ax, 6.5, 4.5, 3.0, 1.0, 'RAG+Metadata\n(enriched context)', '#059669')
draw_arrow(ax, 2.5, 6.5, 1.75, 5.5)
draw_arrow(ax, 5.0, 6.5, 4.75, 5.5)
draw_arrow(ax, 7.5, 6.5, 8.0,  5.5)

# LLM
draw_box(ax, 1.5, 3.0, 7.0, 1.0, 'Open-Source LLM via OpenRouter API\n(7 models evaluated)', '#7C3AED')
draw_arrow(ax, 1.75, 4.5, 3.0, 4.0)
draw_arrow(ax, 4.75, 4.5, 5.0, 4.0)
draw_arrow(ax, 8.0,  4.5, 7.0, 4.0)

# Output
draw_box(ax, 2.5, 1.5, 5.0, 1.0, 'PrecisionRecommendation\n{precision, reasoning}', '#DC2626')
draw_arrow(ax, 5.0, 3.0, 5.0, 2.5)

# Evaluation
draw_box(ax, 1.5, 0.2, 7.0, 0.9, 'Evaluation: Exact-match against RAG4SPEC (465 scenarios)', '#374151')
draw_arrow(ax, 5.0, 1.5, 5.0, 1.1)

# ---- MimirRAG Pipeline (right) ----
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('MimirRAG Pipeline (Reference)', fontsize=13, fontweight='bold', pad=15)

ax.text(5, 11.5, 'PRE-RETRIEVAL', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 0.5, 10.0, 3.5, 1.0, 'PDF Financial\nFilings', '#1D4ED8')
draw_box(ax, 4.5, 10.0, 5.0, 1.0, 'Docling + Chonkie\nChunking & OCR', '#1D4ED8')
draw_box(ax, 0.5,  8.5, 3.5, 1.0, 'Extractor Agent\n(LLM metadata gen)', '#1E40AF')
draw_box(ax, 4.5,  8.5, 5.0, 1.0, 'LanceDB + snowflake\narctic-embed-m-v2.0', '#1E40AF')
draw_arrow(ax, 2.25, 10.0, 2.25, 9.5)
draw_arrow(ax, 7.0,  10.0, 7.0, 9.5)

ax.text(5, 8.0, 'RETRIEVAL (Agentic)', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 0.5, 6.5, 2.0, 1.0, 'Planner\nAgent', '#059669')
draw_box(ax, 3.0, 6.5, 2.5, 1.0, 'Search Agent\n(Hybrid RRF)', '#047857')
draw_box(ax, 6.0, 6.5, 3.5, 1.0, 'Validator\nAgent', '#065F46')
draw_arrow(ax, 2.5, 7.0, 3.0, 7.0)
draw_arrow(ax, 5.5, 7.0, 6.0, 7.0)
draw_arrow(ax, 2.25, 8.5, 2.25, 7.5)

ax.text(5, 6.0, 'POST-RETRIEVAL', ha='center', fontsize=9, color='#6B7280', style='italic')
draw_box(ax, 2.0, 4.5, 6.0, 1.0, 'Writer Agent\n(Citation-grounded generation)', '#7C3AED')
draw_arrow(ax, 7.75, 6.5, 6.0, 5.5)

draw_box(ax, 1.5, 3.0, 7.0, 1.0, 'GPT-4.1 / GPT-4.1-mini / GPT-o4-mini\n(Proprietary OpenAI models)', '#9333EA')
draw_arrow(ax, 5.0, 4.5, 5.0, 4.0)

draw_box(ax, 2.5, 1.5, 5.0, 1.0, 'Free-text answer\nwith inline citations', '#DC2626')
draw_arrow(ax, 5.0, 3.0, 5.0, 2.5)

draw_box(ax, 1.5, 0.2, 7.0, 0.9, 'Evaluation: FinanceBench (150 Q&A), Judge Agent (GPT-4o)', '#374151')
draw_arrow(ax, 5.0, 1.5, 5.0, 1.1)

plt.suptitle('Pipeline Architecture Comparison: App2Spec vs MimirRAG', fontsize=14, y=1.01)
plt.tight_layout()
save_fig('fig7_pipeline_architecture')

## 9. Representative RAG Output Example

Prints one clean example for the paper showing the full RAG pipeline output.

In [ ]:
import json
results = json.loads((RESULTS_ROOT / 'deepseek-v4-flash' / 'results.json').read_text())

# Find a good representative example: RAG correct, ZS wrong, clear reasoning
good_examples = [
    r for r in results
    if r.get('rag_match') and not r.get('zeroshot_match')
    and r.get('error') is None
    and len(r.get('rag_reasoning', '')) > 100
]

if good_examples:
    ex = good_examples[0]
    print('REPRESENTATIVE RAG OUTPUT EXAMPLE')
    print('=' * 70)
    print(f"Scenario ID:      {ex['scenario_id']}")
    print(f"Domain:           {ex['domain']}")
    print(f"Expected:         {ex['expected']}")
    print()
    print(f"Zero-shot answer: {ex['zeroshot_precision']}  (INCORRECT)")
    print(f"Zero-shot reasoning: {ex['zeroshot_reasoning'][:200]}...")
    print()
    print(f"Retrieved sources: {ex['retrieved_sources']}")
    print()
    print(f"RAG answer:       {ex['rag_precision']}  (CORRECT)")
    print(f"RAG reasoning:    {ex['rag_reasoning']}")
    print('=' * 70)
    print()
    print('This example shows RAG correcting a zero-shot error by retrieving')
    print('relevant benchmark-precision mappings from the knowledge base.')
else:
    print('No suitable example found.')

## 10. Environment Table

In [ ]:
env = json.loads((RESULTS_ROOT / 'environment.json').read_text())

print('ENVIRONMENT SETUP')
print('=' * 60)
for k, v in env.items():
    print(f'  {k:<35s}: {v}')
print('=' * 60)

# Save as CSV for paper
df_env = pd.DataFrame(list(env.items()), columns=['Component', 'Value'])
df_env.to_csv(RESULTS_ROOT / 'figures' / 'environment_table.csv', index=False)
print('Saved: figures/environment_table.csv')

## 11. Full Results Summary for Professor Review

In [ ]:
print('APP2SPEC — FULL RESULTS SUMMARY FOR PROFESSOR REVIEW')
print('=' * 80)
print()

print('SECTION 1: MODEL ACCURACY (ordered by active parameters)')
print('-' * 80)
display_cols = ['Model', 'Active Params (B)', 'Release', 'N Scored', 'Errors',
                'ZS Accuracy', 'ZS 95% CI', 'RAG Accuracy', 'RAG 95% CI',
                'RAG Lift', 'RAG Hurt', 'Net RAG Effect']
print(df_acc[display_cols].to_string(index=False))
print()

print('SECTION 2: STATISTICAL SIGNIFICANCE (McNemar Test)')
print('-' * 80)
df_mc = df_mcnemar.set_index('Model').reindex(MODEL_ORDER).reset_index()
print(df_mc[['Model', 'RAG Helped (b)', 'RAG Hurt (c)', 'chi2', 'p-value',
             'Significance', "Cohen's g", 'Interpretation']].to_string(index=False))
print()

print('SECTION 3: RAG+METADATA ABLATION (DeepSeek V4 Flash)')
print('-' * 80)
scored = df_meta[df_meta['error'].isna() | (df_meta['error'] == '')]
print(f'  Zero-shot accuracy:    {scored["zeroshot_match"].mean():.1%}')
print(f'  Base-RAG accuracy:     {scored["rag_match"].mean():.1%}')
print(f'  RAG+Metadata accuracy: {scored["rag_meta_match"].mean():.1%}')
print(f'  Delta (Meta vs RAG):   {(scored["rag_meta_match"].mean() - scored["rag_match"].mean())*100:+.1f}pp')
print()

print('SECTION 4: LATENCY')
print('-' * 80)
print(df_latency[['Model', 'ZS Mean (s)', 'RAG Mean (s)', 'RAG Overhead (s)', 'Total Wall (min)']].to_string(index=False))
print()

print('SECTION 5: KEY FINDINGS')
print('-' * 80)
best_rag  = df_acc.loc[df_acc['RAG Accuracy'].str.rstrip('%').astype(float).idxmax(), 'Model']
best_rag_val = df_acc['RAG Accuracy'].str.rstrip('%').astype(float).max()
best_zs   = df_acc.loc[df_acc['ZS Accuracy'].str.rstrip('%').astype(float).idxmax(), 'Model']
best_zs_val  = df_acc['ZS Accuracy'].str.rstrip('%').astype(float).max()
print(f'  Best RAG accuracy:        {best_rag} ({best_rag_val:.1f}%)')
print(f'  Best zero-shot accuracy:  {best_zs} ({best_zs_val:.1f}%)')
print(f'  All 7 models: RAG improvement statistically significant (p<0.001)')
print(f'  Only model with large effect (Cohen g>=0.3): DeepSeek V4 Flash')
print(f'  Largest RAG lift domain: LLM Evaluation (up to +75pp for MiMo V2.5 Pro)')
print(f'  RAG+Metadata vs Base-RAG: +0.9pp overall (SPECfp 2017: +14.3pp)')
print()

print('SECTION 6: FIGURES GENERATED')
print('-' * 80)
for f in sorted(FIG_DIR.iterdir()):
    print(f'  {f.name}')